# 两次变换，怎样合成一次？

依赖：NumPy、Matplotlib；可选 ipywidgets。依次运行各格即可。图内使用英文标签，避免本地缺少中文字体。

我们比较两条路线：先剪切再竖直拉伸，以及先竖直拉伸再剪切。先拿纸算输入 `(2,1)` 的两个结果，运行后再核对。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

A = np.array([[1, 1], [0, 1]])
B = np.array([[1, 0], [0, 2]])
x = np.array([2, 1])
print('先 A:', A @ x, '再 B:', B @ (A @ x))
print('先 B:', B @ x, '再 A:', A @ (B @ x))


## 用两支基找出合成矩阵

先分别写出 `e1`、`e2` 的完整路线。把两支基最终的输出按顺序排成列，得到什么矩阵？下面没有直接调用矩阵乘矩阵来构造它。


In [ ]:
e1 = np.array([1, 0])
e2 = np.array([0, 1])
C = np.column_stack((B @ (A @ e1), B @ (A @ e2)))
print('先 A 再 B 的矩阵:\n', C)
print('分两步:', B @ (A @ x))
print('合成一步:', C @ x)
assert np.array_equal(C, B @ A)


## 跟踪整个网格

先预测单位正方形四个角和两支基的输出，再画图。每张图使用相同的坐标范围和比例；蓝色、橙色箭头分别跟踪两支基，绿色区域跟踪单位正方形，红点跟踪你选的输入。


In [ ]:
COLORS = {'horizontal': '#2878b5', 'vertical': '#d97922',
          'area': '#359969', 'point': '#c53351'}
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1]])
samples = np.linspace(-2, 2, 81)

def draw_grid(ax, matrix, point, title):
    for fixed in range(-2, 3):
        horizontal = np.column_stack((samples, np.full_like(samples, fixed)))
        vertical = np.column_stack((np.full_like(samples, fixed), samples))
        for line, color in ((horizontal, COLORS['horizontal']),
                            (vertical, COLORS['vertical'])):
            output = line @ matrix.T
            ax.plot(output[:, 0], output[:, 1], color=color,
                    alpha=0.4, linewidth=0.8)
    ax.add_patch(Polygon(square @ matrix.T, closed=True,
                         facecolor=COLORS['area'], edgecolor=COLORS['area'],
                         alpha=0.22, linewidth=2))
    for index, color in enumerate((COLORS['horizontal'], COLORS['vertical'])):
        end = matrix[:, index]
        ax.annotate('', xy=end, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color=color,
                                    lw=3 if index == 0 else 2))
        ax.annotate('e' + str(index + 1), xy=end, xytext=(6, 7),
                    textcoords='offset points', color=color, fontsize=10)
    output = matrix @ point
    ax.scatter(*output, color=COLORS['point'], zorder=6, s=40)
    ax.annotate('({:g}, {:g})'.format(*output), xy=output, xytext=(7, -16),
                textcoords='offset points', color=COLORS['point'], fontsize=10)
    ax.axhline(0, color='0.6', lw=0.7)
    ax.axvline(0, color='0.6', lw=0.7)
    ax.set(xlim=(-8, 8), ylim=(-4.5, 4.5), aspect='equal',
           xlabel='coordinate 1', ylabel='coordinate 2', title=title)
    ax.set_xticks([-8, -4, 0, 4, 8])
    ax.set_yticks([-4, -2, 0, 2, 4])

def inspect_composition(k=1, h=2, s=2, t=1, stages=True):
    shear = np.array([[1., k], [0., 1.]])
    stretch = np.array([[1., 0.], [0., h]])
    point = np.array([s, t])
    if stages:
        matrices = (np.eye(2), shear, stretch @ shear)
        titles = ('Input', 'After A: shear', 'After A, then B: stretch')
    else:
        matrices = (stretch @ shear, shear @ stretch)
        titles = ('A then B: BA', 'B then A: AB')
    fig, axes = plt.subplots(1, len(matrices),
                             figsize=(5 * len(matrices), 3.7), dpi=160)
    for ax, matrix, title in zip(axes, matrices, titles):
        draw_grid(ax, matrix, point, title)
    fig.tight_layout()
    plt.show()


In [ ]:
inspect_composition()


## 交换顺序

先拉伸再剪切时，剪切读取的是怎样的高度？预测红点在哪里，再运行下面这格。特别留意橙色基向量和绿色区域。


In [ ]:
inspect_composition(stages=False)


## 每次只改一个参数

保持 `k=1`，把 `h` 从 2 改为 1，再改为 0。先算第二支基的两条路线，然后看两个网格是否相同。

再固定 `k=1,h=2`，把红点的 `t` 设为 0。一个点的输出相同，能说明整个变换相同吗？

有 ipywidgets 时可使用滑块；否则修改下一格函数参数并重新运行。固定的绘图范围覆盖滑块中的网格与红点。


In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    print('未安装 ipywidgets：直接修改下一格的 k、h、s、t。')
else:
    display(widgets.interactive(
        inspect_composition,
        k=widgets.FloatSlider(value=1, min=-1.5, max=1.5, step=0.25,
                              description='shear k', continuous_update=False),
        h=widgets.FloatSlider(value=2, min=0, max=2, step=0.25,
                              description='stretch h', continuous_update=False),
        s=widgets.FloatSlider(value=2, min=-2, max=2, step=0.25,
                              description='input s', continuous_update=False),
        t=widgets.FloatSlider(value=1, min=-2, max=2, step=0.25,
                              description='input t', continuous_update=False),
        stages=widgets.Checkbox(value=False, description='show stages')))


In [ ]:
inspect_composition(k=1, h=1, s=2, t=1, stages=False)


## 把观察变成一个条件

手算一般输入 `(s,t)` 的两条路线。第二种顺序减第一种，横坐标差是不是 `k*(h-1)*t`？下面检查几组数，它们只核对计算，并不能证明所有输入的结论。

要让两个矩阵相同，应要求这个差对所有 `t` 都为零。请自己写出 `k,h` 的条件。


In [ ]:
for k, h, s, t in [(1, 2, 2, 1), (1, 2, 2, 0),
                    (0, 2, 2, 1), (1, 1, 2, 1), (-1, 0, 1, 2)]:
    shear = np.array([[1., k], [0., 1.]])
    stretch = np.array([[1., 0.], [0., h]])
    point = np.array([s, t])
    difference = shear @ (stretch @ point) - stretch @ (shear @ point)
    predicted = np.array([k * (h - 1) * t, 0])
    print('k,h,s,t =', (k, h, s, t), '实际差:', difference, '预测差:', predicted)
    assert np.allclose(difference, predicted)


## 两个坐标变三个，再变一个

第一步 `(s,t) → (s,t,s+t)`，第二步把三个数相加。先直接写出最终输出，再猜合成矩阵的尺寸与两个数。


In [ ]:
expand = np.array([[1, 0], [0, 1], [1, 1]])
add = np.array([[1, 1, 1]])
combined = add @ expand
print('矩阵尺寸:', combined.shape, '合成矩阵:', combined)
assert np.array_equal(combined, np.array([[2, 2]]))


## 合上实验，自己重建

不用乘法公式，解释合成矩阵的每一列如何产生。再找两个不能交换的矩阵，并找出一个交换顺序后输出恰好相同的输入。最后说明：为什么一个输入的核对不够，而检查所有标准基足以确定线性变换？
